# 🛠️ Workshop: MLOps na prática — o ciclo de vida completo de um modelo

Neste laboratório vamos montar, **em miniatura e dentro do Colab**, o mesmo ciclo que sistemas de produção usam
(inspirado no repositório [full-stack-on-prem-cv-mlops](https://github.com/jomariya23156/full-stack-on-prem-cv-mlops)):

```
 Dados ──► Validação ──► Treino ──► Avaliação ──► Registro ──► Deploy (API) ──► Monitoramento
   ▲         (gate)     (MLflow)    (quality gate)  (Registry)    (FastAPI)         (drift)
   └────────────────────────── retreino automático (Prefect) ◄──────────────────────┘
```

| Etapa | No repo de referência | Aqui no Colab |
|---|---|---|
| Versionamento de dados | DVC | hash do dataset (`data_version`) |
| Validação de dados | DeepChecks | checagens simples em Python |
| Rastreamento de experimentos | MLflow | MLflow (SQLite local) |
| Registro de modelos | central storage + MLflow | MLflow Model Registry (alias `champion`) |
| Orquestração | Prefect | Prefect |
| Serviço do modelo | FastAPI + Gunicorn + Nginx | FastAPI + Uvicorn |
| Log de predições | PostgreSQL | SQLite |
| Detecção de drift | Evidently | teste KS (`scipy`) |

**Como usar:** execute as células **em ordem** com `Shift + Enter`. Cada célula leva poucos segundos.

## 0. Setup (~1 min)

In [ ]:
!pip -q install mlflow==3.17.2 prefect==3.9.0

In [ ]:
import os, hashlib, sqlite3, threading, time, warnings, logging
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["PREFECT_SERVER_ANALYTICS_ENABLED"] = "false"
os.environ["DO_NOT_TRACK"] = "1"
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlflow
from mlflow import MlflowClient
logging.getLogger("mlflow").setLevel(logging.ERROR)  # ---> Definido nivel do LOG para ERROR
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")  # --> Banco SQLite Utilizado para testar Local
mlflow.set_experiment("workshop-digits")
client = MlflowClient()
MODEL_NAME = "digits-classifier"
rng = np.random.default_rng(42)
print("MLflow", mlflow.__version__, "pronto")

## 1. Dados: versionar e validar
Modelo bom começa com dado confiável. Usaremos o dataset **digits** (imagens 8×8 de dígitos manuscritos, pixels de 0 a 16) —
um "mini problema de visão computacional", como o classificador de animais do repositório de referência.

- **Versionamento:** calculamos um *hash* dos dados. Mesmo dado ⇒ mesmo hash. É o que o DVC faz por você.
- **Validação:** se o dado vier quebrado, o pipeline **para antes de treinar** (no repo isso é o DeepChecks).

In [ ]:
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

def versao_dados(X, y):
    return hashlib.sha256(X.tobytes() + y.tobytes()).hexdigest()[:10]

def validar_dados(X, y):
    erros = []
    if X.shape[1] != 64:                  erros.append(f"esperava 64 features, veio {X.shape[1]}")
    if np.isnan(X).any():                 erros.append("existem valores nulos")
    if X.min() < 0 or X.max() > 16:       erros.append("pixels fora do intervalo [0, 16]")
    if np.bincount(y).min() < 0.5 * np.bincount(y).mean(): erros.append("classes muito desbalanceadas")
    if erros:
        raise ValueError("Validação falhou: " + "; ".join(erros))
    return True

validar_dados(X_train, y_train)
print("Treino:", X_train.shape, "| Teste:", X_test.shape, "| data_version =", versao_dados(X_train, y_train))

fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, label in zip(axes, X_train, y_train):
    ax.imshow(img.reshape(8, 8), cmap="gray_r"); ax.set_title(label); ax.axis("off")
plt.show()

In [ ]:
# Teste: o que acontece com um dado "quebrado"?
X_ruim = X_train.copy(); X_ruim[0, 0] = np.nan
try:
    validar_dados(X_ruim, y_train)
except ValueError as e:
    print(e)

## 2. Treino rastreado com MLflow
Cada treino vira um **run**: parâmetros, métricas, versão dos dados e o próprio modelo ficam registrados.
Acabou o "qual era mesmo o notebook que deu 97%?".

In [ ]:
MODELOS = {
    "logreg":        lambda p: LogisticRegression(max_iter=2000, C=p.get("C", 1.0)),
    "knn":           lambda p: KNeighborsClassifier(n_neighbors=p.get("k", 5)),
    "svm":           lambda p: SVC(C=p.get("C", 1.0), gamma="scale"),
}

def treinar(tipo, params, X_tr, y_tr, X_ev, y_ev, tags=None):
    with mlflow.start_run(run_name=tipo) as run:
        modelo = MODELOS[tipo](params).fit(X_tr, y_tr)
        pred = modelo.predict(X_ev)
        mlflow.log_params({"tipo": tipo, **params})
        mlflow.set_tags({"data_version": versao_dados(X_tr, y_tr), **(tags or {})})
        mlflow.log_metrics({"accuracy": accuracy_score(y_ev, pred),
                            "f1_macro": f1_score(y_ev, pred, average="macro")})
        mlflow.sklearn.log_model(modelo, name="model", input_example=X_tr[:2])
        return run.info.run_id

for tipo, params in [("logreg", {"C": 0.01}), ("knn", {"k": 15}), ("svm", {"C": 5.0})]:
    treinar(tipo, params, X_train, y_train, X_test, y_test)

runs = mlflow.search_runs(order_by=["metrics.accuracy DESC"])
runs[["tags.mlflow.runName", "params.tipo", "metrics.accuracy", "metrics.f1_macro", "tags.data_version"]]

## 3. Model Registry: qual modelo está "em produção"?
Registramos o melhor run e damos a ele o **alias `champion`**. O serviço de produção nunca aponta para um arquivo solto:
ele pede `models:/digits-classifier@champion`. Trocar o modelo em produção = mover o alias.

In [ ]:
melhor_run = runs.iloc[0].run_id
mv = mlflow.register_model(f"runs:/{melhor_run}/model", MODEL_NAME)
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
print(f" {MODEL_NAME} v{mv.version} agora é o champion (run {melhor_run[:8]})")

## 4. Pipeline automatizado com Prefect
Até aqui fizemos tudo "na mão". Agora empacotamos o ciclo em um **flow** do Prefect com *tasks*:

`validar → treinar → quality gate → promover`

O **quality gate** compara o candidato com o champion atual **no mesmo conjunto de avaliação**.
Só promove se for melhor — é o "teste automatizado" do mundo de ML.

In [ ]:
from prefect import flow, task, get_run_logger

@task
def t_validar(X, y):
    return validar_dados(X, y)

@task
def t_treinar(X_tr, y_tr, X_ev, y_ev, motivo):
    return treinar("svm", {"C": 5.0}, X_tr, y_tr, X_ev, y_ev, tags={"motivo": motivo})

@task
def t_quality_gate(run_id, X_ev, y_ev, minimo=0.90):
    log = get_run_logger()
    candidato = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
    acc_cand = accuracy_score(y_ev, candidato.predict(X_ev))
    try:
        champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
        acc_champ = accuracy_score(y_ev, champion.predict(X_ev))
    except Exception:
        acc_champ = 0.0
    log.info(f"candidato={acc_cand:.3f} | champion={acc_champ:.3f} | mínimo={minimo}")
    return acc_cand >= minimo and acc_cand > acc_champ

@task
def t_promover(run_id):
    mv = mlflow.register_model(f"runs:/{run_id}/model", MODEL_NAME)
    client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
    return mv.version

@flow(name="pipeline-treino")
def pipeline_treino(X_tr, y_tr, X_ev, y_ev, motivo="manual"):
    log = get_run_logger()
    t_validar(X_tr, y_tr)
    run_id = t_treinar(X_tr, y_tr, X_ev, y_ev, motivo)
    if t_quality_gate(run_id, X_ev, y_ev):
        versao = t_promover(run_id)
        log.info(f"🚀 Promovido: v{versao}")
        return versao
    log.warning("⛔ Reprovado no quality gate — champion mantido")
    return None

# Mesmo dado, mesmo modelo: o candidato NÃO é melhor que o champion ⇒ deve ser reprovado.
pipeline_treino(X_train, y_train, X_test, y_test, motivo="teste-gate")

## 5. Deploy: o modelo vira uma API
Subimos um serviço **FastAPI** (como o `dl_service` do repo) que:
- carrega o `champion` do Registry;
- responde `POST /predict`;
- **grava cada requisição** em um banco (aqui SQLite; no repo, PostgreSQL) — sem isso não existe monitoramento;
- expõe `PUT /reload` para trocar de modelo **sem derrubar o serviço** (mesma ideia do `PUT /update_model` do repo).

In [ ]:
import json, requests, uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

DB = "predicoes.db"
with sqlite3.connect(DB) as con:
    con.execute("CREATE TABLE IF NOT EXISTS predicoes (ts REAL, versao TEXT, entrada TEXT, predicao INT)")

app = FastAPI(title="Digits service")
estado = {}

def carregar_champion():
    mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
    estado["modelo"] = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
    estado["versao"] = mv.version
carregar_champion()

class Entrada(BaseModel):
    pixels: list[float]          # 64 valores (imagem 8x8)

@app.get("/health")
def health():
    return {"status": "ok", "modelo": MODEL_NAME, "versao": estado["versao"]}

@app.post("/predict")
def predict(e: Entrada):
    pred = int(estado["modelo"].predict([e.pixels])[0])
    with sqlite3.connect(DB) as con:
        con.execute("INSERT INTO predicoes VALUES (?,?,?,?)", (time.time(), estado["versao"], json.dumps(e.pixels), pred))
    return {"predicao": pred, "versao_modelo": estado["versao"]}

@app.put("/reload")
def reload():
    carregar_champion()
    return {"status": "recarregado", "versao": estado["versao"]}

servidor = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=servidor.run, daemon=True).start()
time.sleep(2)
API = "http://127.0.0.1:8000"
print(requests.get(f"{API}/health").json())

In [ ]:
# Uma chamada real à API — exatamente como um app/cliente faria
amostra = X_test[0]
print("Esperado:", y_test[0], "| Resposta:", requests.post(f"{API}/predict", json={"pixels": amostra.tolist()}).json())

## 6. Produção muda: simulando *data drift*
O modelo foi treinado com imagens "limpas". Imagine que trocaram o scanner/câmera: as imagens agora chegam
**mais escuras e com ruído**. Ninguém avisou o time de ML. O modelo continua respondendo... mas errando.

In [ ]:
def camera_nova(X):
    # imagens mais claras/lavadas + ruído: o mundo real mudou
    return np.clip(X * 0.5 + 6 + rng.normal(0, 2.5, X.shape), 0, 16)

X_prod_ok, y_prod_ok = X_test[:150], y_test[:150]
X_prod_drift, y_prod_drift = camera_nova(X_test[150:]), y_test[150:]

def enviar_trafego(X_lote):
    return [requests.post(f"{API}/predict", json={"pixels": x.tolist()}).json()["predicao"] for x in X_lote]

pred_ok = enviar_trafego(X_prod_ok)
pred_drift = enviar_trafego(X_prod_drift)
print(f"Acurácia antes da câmera nova:  {accuracy_score(y_prod_ok, pred_ok):.1%}")
print(f"Acurácia depois da câmera nova: {accuracy_score(y_prod_drift, pred_drift):.1%}  ⚠️")

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i in range(8):
    axes[0, i].imshow(X_test[150 + i].reshape(8, 8), cmap="gray_r"); axes[0, i].axis("off")
    axes[1, i].imshow(X_prod_drift[i].reshape(8, 8), cmap="gray_r"); axes[1, i].axis("off")
axes[0, 0].set_title("antes", loc="left"); axes[1, 0].set_title("câmera nova", loc="left")
plt.show()

### Monitoramento: detectando drift **sem precisar dos rótulos**
Em produção quase nunca temos o rótulo na hora. Então comparamos a **distribuição das entradas** recentes
com a do treino (referência), pixel a pixel, usando o teste de Kolmogorov-Smirnov.
Se muitas features mudaram de distribuição ⇒ **alerta de drift**. (É isso que o Evidently calcula no repo.)

In [ ]:
from scipy.stats import ks_2samp

def ler_producao(ultimas=100):
    with sqlite3.connect(DB) as con:
        df = pd.read_sql(f"SELECT entrada FROM predicoes ORDER BY ts DESC LIMIT {ultimas}", con)
    return np.array([json.loads(e) for e in df.entrada])

def checar_drift(X_ref, X_atual, alpha=0.05, limite=0.3):
    pvalores = np.array([ks_2samp(X_ref[:, j], X_atual[:, j]).pvalue for j in range(X_ref.shape[1])
                         if X_ref[:, j].std() > 0])
    frac = float((pvalores < alpha).mean())
    return frac > limite, frac

for nome, janela in [("antes", X_prod_ok[:100]), ("câmera nova", ler_producao(100))]:
    drift, frac = checar_drift(X_train, janela)
    print(f"Janela '{nome}': drift={drift} ({frac:.0%} das features mudaram de distribuição)")

## 7. Fechando o ciclo: monitoramento ➜ retreino ➜ deploy automático
Agora o flow de **monitoramento** decide sozinho:
1. lê as últimas predições do banco;
2. se detectar drift, junta dados novos **rotulados** (na vida real: rotulação humana/feedback) ao histórico;
3. dispara o `pipeline_treino` (com quality gate!);
4. se houver novo champion, chama `PUT /reload` na API.

> Em produção esse flow roda **agendado** (ex.: `monitorar_e_retreinar.serve(cron="0 * * * *")`) — no repo ele roda semanalmente.

In [ ]:
# Dados novos rotulados (simulação do feedback de produção), separados em treino e avaliação
X_novo, y_novo = camera_nova(X_train), y_train
X_novo_tr, X_novo_ev, y_novo_tr, y_novo_ev = train_test_split(X_novo, y_novo, test_size=0.3, random_state=0)

@flow(name="monitorar-e-retreinar")
def monitorar_e_retreinar():
    log = get_run_logger()
    drift, frac = checar_drift(X_train, ler_producao(100))
    log.info(f"Drift={drift} ({frac:.0%} das features)")
    if not drift:
        return "sem drift — nada a fazer"
    # avaliação justa: cenário antigo + cenário novo
    X_tr = np.vstack([X_train, X_novo_tr]); y_tr = np.concatenate([y_train, y_novo_tr])
    X_ev = np.vstack([X_test, X_novo_ev]);  y_ev = np.concatenate([y_test, y_novo_ev])
    versao = pipeline_treino(X_tr, y_tr, X_ev, y_ev, motivo="drift-detectado")
    if versao:
        log.info(f"Deploy: {requests.put(f'{API}/reload').json()}")
    return versao

monitorar_e_retreinar()

In [ ]:
# A API está servindo o novo modelo? Reenviamos o mesmo tráfego da câmera nova:
print(requests.get(f"{API}/health").json())
pred_drift_v2 = enviar_trafego(X_prod_drift)
print(f"Acurácia na câmera nova — antes: {accuracy_score(y_prod_drift, pred_drift):.1%} | depois do retreino: {accuracy_score(y_prod_drift, pred_drift_v2):.1%} ✅")

## 8. Auditoria: quem está em produção e por quê?
Tudo ficou rastreado: cada versão aponta para o run, que aponta para parâmetros, métricas, dados e o motivo do treino.

In [ ]:
versoes = client.search_model_versions(f"name='{MODEL_NAME}'")
champion = client.get_model_version_by_alias(MODEL_NAME, "champion").version
pd.DataFrame([{
    "versao": v.version,
    "champion": "⭐" if v.version == champion else "",
    "motivo": client.get_run(v.run_id).data.tags.get("motivo", "manual"),
    "data_version": client.get_run(v.run_id).data.tags.get("data_version"),
    "accuracy (avaliação)": round(client.get_run(v.run_id).data.metrics["accuracy"], 3),
} for v in sorted(versoes, key=lambda v: int(v.version))])

### (Opcional) Abrir a interface do MLflow
Rode a célula abaixo e clique no link gerado para navegar pelos experimentos, runs e modelos registrados.

In [ ]:
import subprocess
# porta 5050 = a mesma do MLflow no repo de referência. "--allowed-hosts '*'" libera o proxy do Colab (VM descartável).
subprocess.Popen("mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5050 --allowed-hosts '*' > mlflow_ui.log 2>&1",
                 shell=True)
for _ in range(30):
    try:
        requests.get("http://127.0.0.1:5050/health", timeout=1); break
    except requests.ConnectionError:
        time.sleep(1)
try:
    from google.colab import output
    output.serve_kernel_port_as_window(5050, path="/")
except ImportError:
    print("Fora do Colab: abra http://localhost:5050")

## 🧩 Desafios (para depois do workshop)
1. Troque o `minimo` do quality gate para `0.99`. O que acontece no retreino?
2. Deixe a `camera_nova` mais suave (ex.: `X * 0.9 + 1`). O monitor ainda detecta drift?
3. Adicione ao `t_treinar` uma busca entre `logreg`, `knn` e `svm` e promova o melhor.
4. Agende o `monitorar_e_retreinar` com `.serve(cron=...)` e veja os runs na UI do Prefect.
5. Leia o `docker-compose.yml` do [repo de referência](https://github.com/jomariya23156/full-stack-on-prem-cv-mlops) e mapeie cada serviço para uma etapa deste notebook.